# 08 · Ensembles, and the leaderboard across all model families

Does combining the best models beat the best single model? Runs:

| Run | What |
|---|---|
| EN-01 | Soft vote: L2 logistic regression + monotone LightGBM |
| EN-02 | Soft vote: logistic regression + monotone WoE scorecard + monotone LightGBM |
| EN-03 | Stacking: logistic regression, extra trees, monotone LightGBM → logistic meta-model on out-of-fold predictions |
| EN-04 | Bagged logistic regression (50 bootstrap fits; stability rather than accuracy) |

All on `FS_TRANSFER`, the features that also exist in Salud's real data. Base models use **fixed, sensible
settings chosen up front** (not the values tuned on all training rows, which would let the test folds leak into
the settings).

In [ ]:
%matplotlib inline
import warnings; warnings.filterwarnings("ignore")
import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from scipy.stats import loguniform, uniform, randint
from salud_pipeline.phase2.propensity import modeling as M
sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 30); pd.set_option("display.width", 200); pd.set_option("display.precision", 3)

DESIGN = "fixed"   # training rows: "fixed" (one per patient on the scoring date; notebook 00 §6) or "call"
WORLD = "salud"    # synthetic world: "salud" (tuned to Salud's data, salud-synth-calibrate) or "original"
dev, holdout, real = M.load_data(design=DESIGN, world=WORLD)
SPLITS = M.cv_splits(dev)          # the same 25 folds for every run in every notebook -> paired comparisons
NB = "08"
def run(rid, pipe, fs, desc, **kw):
    return M.run_experiment(rid, pipe, dev, fs, notebook=NB, description=desc, splits=SPLITS, **kw)
print(f"{DESIGN} design: {len(dev):,} training rows, pay rate {dev.paid_30d.mean():.2f}; 25 folds")

In [ ]:
from sklearn.ensemble import VotingClassifier, StackingClassifier, BaggingClassifier, ExtraTreesClassifier
from sklearn.model_selection import StratifiedKFold
F = M.FS_TRANSFER
LR = lambda: M.pipeline(M.preprocess("z"), M.logreg(C=0.1))
SC = lambda: M.pipeline([("woe", M.WoE(monotonic=True))], M.logreg(C=1.0))
GB = lambda: M.pipeline(M.preprocess("none"), M.lgbm(F, monotone=True, num_leaves=7, learning_rate=.03, n_estimators=300,
                                                      min_child_samples=40, subsample=.8, subsample_freq=1, colsample_bytree=.8))
ET = lambda: M.pipeline(M.preprocess("none"), ExtraTreesClassifier(400, min_samples_leaf=20, max_features=.5, random_state=M.SEED, n_jobs=1))
run("EN-00a", LR(), "FS_TRANSFER", "Base: L2 logistic regression (C=0.1, z-score)")
run("EN-00b", SC(), "FS_TRANSFER", "Base: monotone WoE scorecard (smart bins)")
run("EN-00c", GB(), "FS_TRANSFER", "Base: monotone LightGBM (fixed settings)")
run("EN-00d", ET(), "FS_TRANSFER", "Base: extra trees (min leaf 20)")
run("EN-01", VotingClassifier([("lr", LR()), ("gb", GB())], voting="soft"), "FS_TRANSFER", "Soft vote: LR + LightGBM")
run("EN-02", VotingClassifier([("lr", LR()), ("sc", SC()), ("gb", GB())], voting="soft"), "FS_TRANSFER", "Soft vote: LR + scorecard + LightGBM")
run("EN-03", StackingClassifier([("lr", LR()), ("et", ET()), ("gb", GB())], final_estimator=M.logreg(C=1.0),
                                cv=StratifiedKFold(5, shuffle=True, random_state=M.SEED), stack_method="predict_proba"),
    "FS_TRANSFER", "Stacking: LR + extra trees + LightGBM -> logistic meta-model")
run("EN-04", M.pipeline(M.preprocess("z"), BaggingClassifier(M.logreg(C=0.1), n_estimators=50, random_state=M.SEED)), "FS_TRANSFER",
    "Bagged logistic regression (50 bootstrap fits)");

In [ ]:
board = M.leaderboard(notebook=NB)
cols = ["run_id", "description", "feature_set", "auc", "auc_sd", "brier", "cal_slope", "top20", "n_features", "fit_s"]
board[cols]

In [ ]:
r = M.read_results(); r = r[r.notebook == NB]
plt.figure(figsize=(10, 0.35 * r.run_id.nunique() + 1.2))
sns.boxplot(data=r, y="run_id", x="auc", order=board.run_id, color="#8DA0CB")
plt.axvline(M.leaderboard().query("run_id == 'REF-LR'").auc.iloc[0], ls="--", c="#C44E52", label="REF-LR (notebook 01)")
plt.legend(); plt.title("AUC across the same 25 folds"); plt.ylabel(""); plt.show()

In [ ]:
best = board.run_id.iloc[0]
pd.DataFrame([M.corrected_ttest(best, rid) for rid in board.run_id[1:]]).round(4)

## Leaderboard across every notebook (pre-call target, fixed-date design)
Rules, oracle and the call-design diagnostics are shown for reference. Statistical comparison: each run against the
best, with the Nadeau–Bengio corrected t-test (fold scores overlap, so a plain t-test would be over-confident).

In [ ]:
allb = M.leaderboard()
ref = allb[~allb.run_id.str.contains("ORACLE|B-05|V-0|D-|REF")].reset_index(drop=True)
top = ref.run_id.iloc[0]
ref["p vs best"] = [np.nan if rid == top else M.corrected_ttest(top, rid)["p_value"] for rid in ref.run_id]
ref["family"] = ref.run_id.str.extract(r"^([A-Z]+)")[0].map({"LR": "Logistic regression", "SC": "Scorecard", "DT": "Decision tree",
    "RF": "Random forest / extra trees", "GB": "Gradient boosting", "OT": "Other (k-NN, SVM, NB, MLP)", "EN": "Ensemble", "B": "Rule baseline"})
ref[["run_id", "family", "description", "feature_set", "auc", "auc_se", "p vs best", "brier", "cal_slope", "n_features"]].head(25)

In [ ]:
fam = ref.dropna(subset=["family"]).sort_values("auc", ascending=False).groupby("family").head(1).sort_values("auc")
plt.figure(figsize=(9, 4)); plt.barh(fam.family, fam.auc, xerr=fam.auc_se * 1.96, color="#4C72B0")
plt.axvline(allb.query("run_id == 'B-05'").auc.iloc[0], ls="--", c="grey", label="oracle (hidden traits)")
plt.xlim(.5, .82); plt.legend(); plt.title("Best run per model family (95% CI of the mean)"); plt.xlabel("AUC"); plt.show()
fam[["family", "run_id", "description", "auc", "brier"]]

## One-standard-error pick (simplest model within 1 SE of the best)

In [ ]:
cand = ref[ref.family.notna() & (ref.family != "Rule baseline")]
pick = M.one_se_pick(cand)
print(f"Best AUC: {cand.auc.max():.3f} ({cand.run_id.iloc[0]}); 1-SE line: {cand.auc.max() - cand.auc_se.iloc[0]:.3f}")
print(f"Pick: {pick.run_id} — {pick.description} (AUC {pick.auc:.3f}, {pick.n_features:.0f} features, Brier {pick.brier:.3f})")
cand[cand.auc >= cand.auc.max() - cand.auc_se.iloc[0]][["run_id", "description", "feature_set", "auc", "n_features", "brier"]]

## Findings (Salud-tuned world)
- Stacking (EN-03, 0.880) and soft voting (EN-02, 0.879) are no better than their best base model (extra trees, 0.879; p ≥ 0.7).
  The base models make the same mistakes, so combining them doesn't help.
- Across families, the top runs all lie within 0.887–0.872. On cross-validation alone, the one-SE rule picks **RF-04** (6 features).
  Notebook 11 makes the final choice among the transferable finalists.